# Notebook 14: Sequential Multi-Agent Investment System
## Agent 1 → Agent 2 → Agent 3

A **Router → Sequential Orchestrator** that chains the three investment agents
in the correct order, passing live context between them.

## Key Change from Previous Version

| | Before | Now |
|--|--------|-----|
| Portfolio data | Hardcoded `SAMPLE_PORTFOLIOS` list (25 holdings) | Loaded dynamically from **NB12 output** (`consolidated_portfolio.json`) |
| Analytics engine | Duplicated `consolidate_holdings` + `compute_analytics` | Reads pre-computed analytics from NB12's JSON |
| Agent 2 prompt | Hardcoded "$1,026,834 AUM across 5 accounts" | Built at runtime from loaded portfolio |
| Agent 3 prompt | Hardcoded ticker list | Built at runtime — same `load_portfolio_context()` pattern as NB13 |
| Agent 2 save tool | Wrote its own schema | Writes NB12-compatible `consolidated_portfolio.json` schema |

## Architecture
```
consolidated_portfolio.json   (output of Notebook 12)
          │
          ▼
  load_portfolio_context()    ← shared helper (same as NB13)
          │
          ├── Agent 2 system prompt  (live AUM, holdings count, source files)
          └── Agent 3 system prompt  (full holdings table with allocations)
                    │
                    ▼
            User Query
                │
                ▼
          ┌─────────────┐
          │ Router (LLM)│  → route string
          └──────┬──────┘
                 │
     ┌───────────▼──────────────────────────┐
     │     Sequential Orchestrator           │
     │  agent1 → agent2 → agent3            │
     │  Context passed between each step    │
     └───────────────────────────────────────┘
                 │
    ┌────────────┼────────────┐
    ▼            ▼            ▼
 Agent 1      Agent 2      Agent 3
 Designer     Analyst      Educator
 (NB11)       (NB12)       (NB13)
```

## Three-Agent System
| Notebook | Agent | Role | Input | Output |
|----------|-------|------|-------|--------|
| 11 | Agent 1 | Profile investor, design portfolio | Conversation | `portfolio.json` |
| 12 | Agent 2 | Consolidate & analyse portfolios | Portfolio folder | `consolidated_portfolio.json` |
| 13 | Agent 3 | Investment education | `consolidated_portfolio.json` | Education chat |
| **14** | **Orchestrator** | **Route & chain all three** | **Any query** | **Best-agent response** |


## How to Use This Notebook

### Prerequisites
1. **Run Notebook 12 first** to produce `consolidated_portfolio.json`.  
   Without it the system falls back to generic mode (warns you clearly).
2. A `.env` file at `../.env`:
   ```
   LLM_PROVIDER=openai
   LLM_MODEL=gpt-4.1-mini
   OPENAI_API_KEY=your-key
   SERPER_API_KEY=your-key
   ```

### Configuration (two variables)
```python
PORTFOLIO_FOLDER            = "../data/user1"
CONSOLIDATED_PORTFOLIO_FILE = "../data/outputs/consolidated_portfolio.json"
```

### Running
Use **Kernel > Restart & Run All**.  The notebook will:
1. Load your consolidated portfolio from NB12
2. Build personalised system prompts for Agents 2 & 3
3. Run 5 test queries that exercise the full agent chain
4. Open the interactive chat for free-form questions

### Routing Logic
| Query type | Route | Agents invoked |
|------------|-------|----------------|
| Build / design a portfolio | `agent1` | Agent 1 only |
| Analytics question | `agent2` | Agent 2 only |
| Concept explanation | `agent3` | Agent 3 only |
| Profile + analytics | `agent1+agent2` | 1 → 2 |
| Analytics + explanation | `agent2+agent3` | 2 → 3 |
| Full pipeline | `agent1+agent2+agent3` | 1 → 2 → 3 |


## Installation

In [ ]:
%pip install langchain langchain-openai langchain-community langgraph \
             python-dotenv google-search-results \
             pandas openpyxl PyMuPDF --quiet

print("Packages installed")


## Imports & Environment

In [2]:
import json
import os
from pathlib import Path
from typing import Literal

import pandas as pd
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool
from langchain_community.utilities import GoogleSerperAPIWrapper
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, MessagesState
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.memory import MemorySaver

load_dotenv()
load_dotenv('../.env')

try:
    from ai_course_utils import load_llm_from_env, display_config
    display_config()
except ImportError:
    def load_llm_from_env(temperature: float = 0.0):
        return ChatOpenAI(model=os.getenv('LLM_MODEL', 'gpt-4o-mini'), temperature=temperature)
    print(f"LLM_MODEL : {os.getenv('LLM_MODEL','gpt-4o-mini')}")
    print(f"OpenAI key: {'Set ✅' if os.getenv('OPENAI_API_KEY') else 'NOT SET ⚠️'}")
    print(f"Serper key: {'Set ✅' if os.getenv('SERPER_API_KEY') else 'NOT SET ⚠️'}")

DIVIDER = "=" * 70
print("Imports successful")


NS: what it reads is:  gpt-4.1-mini
API CONFIGURATION (.env file)
LLM Provider:    openai
LLM Model:       gpt-4.1-mini
Temperature:     0.0

API Keys Status:
  OpenAI               ✓ Set
  Google               ✗ Not set
  Mistral              ✗ Not set
  Anthropic            ✗ Not set
  Serper (Web Search)  ✓ Set

Data Files:
  Provide file paths as function parameters
  Example: load_use_case_config('your_file.xlsx')
Imports successful


## Configuration

Two variables to set — folder for Agent 1 to read, and the NB12 output file
that Agents 2 & 3 use to personalise their responses.


In [3]:
# ── USER-DEFINED SETTINGS ─────────────────────────────────────────────────────

# Folder of portfolio files for Agent 1 / Agent 2 consolidation.
PORTFOLIO_FOLDER = "../data/input/user1"

# JSON produced by Notebook 12 — used to personalise Agents 2 & 3.
CONSOLIDATED_PORTFOLIO_FILE = "../data/outputs/consolidated_portfolio.json"

# Where Agent 1 saves new portfolios (passed to Agent 2 for consolidation).
AGENT1_OUTPUT_FILE = "../data/outputs/portfolio.json"

# ── SHARED IN-MEMORY STATE ────────────────────────────────────────────────────
SHARED_STATE = {
    "investor_profile"    : None,   # set by Agent 1
    "portfolio"           : None,   # set by Agent 1
    "conversation_history": [],     # cross-agent context window
}

print(f"Portfolio folder  : {os.path.abspath(PORTFOLIO_FOLDER)}")
print(f"Consolidated file : {os.path.abspath(CONSOLIDATED_PORTFOLIO_FILE)}")
print(f"Agent 1 output    : {os.path.abspath(AGENT1_OUTPUT_FILE)}")


Portfolio folder  : /Users/nsathi/Library/CloudStorage/Dropbox/Applied AI Institute/Courses/GenerativeAI/G-AI-5-Building-Full-Scale-AI-Solutions/AI-Sandbox/ai-powered-move-recommendation/data/input/user1
Consolidated file : /Users/nsathi/Library/CloudStorage/Dropbox/Applied AI Institute/Courses/GenerativeAI/G-AI-5-Building-Full-Scale-AI-Solutions/AI-Sandbox/ai-powered-move-recommendation/data/outputs/consolidated_portfolio.json
Agent 1 output    : /Users/nsathi/Library/CloudStorage/Dropbox/Applied AI Institute/Courses/GenerativeAI/G-AI-5-Building-Full-Scale-AI-Solutions/AI-Sandbox/ai-powered-move-recommendation/data/outputs/portfolio.json


## Load Portfolio from Notebook 12

This helper is **identical in design to Notebook 13** — it reads
`consolidated_portfolio.json` and builds a context block that is injected
into Agent 2 and Agent 3 system prompts at runtime.

No ticker, allocation, or dollar value is hardcoded anywhere below.


In [4]:
def load_portfolio_context(filepath: str) -> dict:
    """
    Load consolidated_portfolio.json produced by Notebook 12.

    Returns:
        holdings        : list of holding dicts
        context_block   : formatted string for system prompt injection
        analytics_block : analytics summary (return, AUM, asset breakdown)
        metadata        : source info
        personalised    : True if a real portfolio was loaded
    """
    path = Path(filepath)

    # ── Graceful fallback if NB12 has not been run ─────────────────────────
    if not path.exists():
        print(f"⚠  Portfolio file not found: {path.resolve()}")
        print("   Run Notebook 12 first to enable personalised agent responses.")
        print("   Agents 2 & 3 will run in GENERIC mode.")
        return {
            "holdings":       [],
            "context_block":  "No portfolio loaded. Provide general investment guidance.",
            "analytics_block":"No analytics available.",
            "metadata":       {"source": "none", "n_holdings": 0, "total_value": None,
                               "source_files": []},
            "personalised":   False,
        }

    with open(path) as f:
        data = json.load(f)

    holdings = data.get("holdings", [])
    if not holdings:
        print(f"⚠  Portfolio file is empty: {path.name}")
        return {
            "holdings":       [],
            "context_block":  "Portfolio file was empty.",
            "analytics_block":"No analytics available.",
            "metadata":       {"source": path.name, "n_holdings": 0, "total_value": None,
                               "source_files": []},
            "personalised":   False,
        }

    # ── Holdings context block ──────────────────────────────────────────────
    total_value   = data.get("total_value_usd")
    source_files  = data.get("source_files", [path.name])

    lines = [
        "INVESTOR'S CONSOLIDATED PORTFOLIO",
        "-" * 54,
    ]
    if total_value:
        lines.append(f"  Total AUM : ${total_value:,.0f}")
    lines.append(f"  Sources   : {', '.join(source_files)}")
    lines.append(f"  Holdings  : {len(holdings)}")
    lines.append("-" * 54)

    for h in holdings:
        ticker   = h.get("ticker", "?")
        name     = h.get("company_name", ticker)
        inv_type = h.get("investment_type", "Unknown")
        alloc    = h.get("allocation_pct", 0)
        value    = h.get("value_usd")
        val_str  = f"  (${value:>10,.0f})" if value else ""
        lines.append(
            f"  {ticker:<8} | {inv_type:<14} | {alloc:6.2f}%{val_str}  — {name}"
        )

    lines.append("-" * 54)
    context_block = "\n".join(lines)

    # ── Analytics block (asset-type breakdown + totals) ─────────────────────
    type_totals: dict[str, float] = {}
    for h in holdings:
        t = h.get("investment_type", "Unknown")
        type_totals[t] = type_totals.get(t, 0) + h.get("allocation_pct", 0)

    alines = ["ASSET TYPE BREAKDOWN", "-" * 30]
    for t, pct in sorted(type_totals.items(), key=lambda x: -x[1]):
        alines.append(f"  {t:<18} {pct:6.2f}%")
    if total_value:
        alines.append(f"  {'Total AUM':<18} ${total_value:,.0f}")
    analytics_block = "\n".join(alines)

    metadata = {
        "source":       path.name,
        "n_holdings":   len(holdings),
        "total_value":  total_value,
        "source_files": source_files,
    }

    return {
        "holdings":       holdings,
        "context_block":  context_block,
        "analytics_block": analytics_block,
        "metadata":       metadata,
        "personalised":   True,
    }


# ── Load on startup ─────────────────────────────────────────────────────────
portfolio_data = load_portfolio_context(CONSOLIDATED_PORTFOLIO_FILE)

print(DIVIDER)
if portfolio_data["personalised"]:
    print("PORTFOLIO LOADED — agents will be personalised")
    print(DIVIDER)
    print(portfolio_data["context_block"])
else:
    print("GENERIC MODE — run Notebook 12 for personalised agents")
print(DIVIDER)


PORTFOLIO LOADED — agents will be personalised
INVESTOR'S CONSOLIDATED PORTFOLIO
------------------------------------------------------
  Total AUM : $979,963
  Sources   : Portfolio1.pdf, Portfolio2.pdf, Portfolio3.pdf, Portfolio4.pdf, Portfolio5.pdf
  Holdings  : 17
------------------------------------------------------
  VTI      | ETF            |  20.85%  ($   204,294)  — Vanguard Total Stock Market ETF
  NVDA     | Stock          |  12.48%  ($   122,321)  — NVIDIA
  BND      | Bond ETF       |  11.13%  ($   109,095)  — Vanguard Total Bond Market ETF
  MSFT     | Stock          |  10.94%  ($   107,207)  — Microsoft
  AAPL     | Stock          |   8.57%  ($    84,022)  — Apple
  IJR      | ETF            |   6.20%  ($    60,742)  — iShares Core S&P Small-Cap ETF
  VEA      | ETF            |   5.89%  ($    57,752)  — Vanguard FTSE Developed Markets ETF
  VTTHX    | Other          |   4.05%  ($    39,681)  — Vanguard Target Retirement 2035
  LQD      | Bond ETF       |   3.68%  ($  

## Build Agent System Prompts

All three prompts are built at runtime from `portfolio_data`.
No ticker, AUM figure, or allocation percentage is hardcoded.


In [5]:
def build_agent_prompts(portfolio_ctx: dict) -> tuple[str, str, str]:
    """
    Build system prompts for all three agents from the loaded portfolio context.

    Returns (agent1_prompt, agent2_prompt, agent3_prompt).
    Called once at startup; re-call after reload_portfolio() if needed.
    """
    context_block   = portfolio_ctx["context_block"]
    analytics_block = portfolio_ctx["analytics_block"]
    personalised    = portfolio_ctx["personalised"]
    metadata        = portfolio_ctx["metadata"]

    # ── Agent 1: Portfolio Designer ─────────────────────────────────────────
    a1 = """\
You are an investment coach whose mission is to help investors manage their
investments wisely. Use the questionnaire below to engage the investor in a
warm, friendly conversation and build a complete Investor Profile.

QUESTIONNAIRE TOPICS:
Q1 – Goal (retirement/purchase/growth/income/education)
Q2 – Experience (beginner/intermediate/advanced)
Q3 – Reaction to 15% drop (sell all/sell some/hold/buy more)
Q4 – Products familiar with (stocks/ETFs/crypto/PE)
Q5 – Liquidity needs (< 2yr / 2-5yr / 6-10yr / 10+ yr)
Q6 – Philosophy (safety-first/balanced/growth-oriented)

Guidelines:
- Ask 2-3 questions per turn. Be conversational and encouraging.
- Once you have enough context, recommend a portfolio of 8-12 holdings
  (stocks, ETFs, bonds) with allocations summing to 100%.
- When asked about market data, use the search_web tool.
- After recommending the portfolio, call portfolio_generation to save it.
- This is for educational purposes only, not financial advice.
"""

    # ── Agent 2: Consolidator & Analyst ────────────────────────────────────
    if personalised:
        n          = metadata["n_holdings"]
        aum_str    = (f"${metadata['total_value']:,.0f}" if metadata["total_value"]
                      else f"{n} holdings")
        files_str  = ", ".join(metadata["source_files"])
        a2_ctx = f"""
You have access to the investor's consolidated portfolio ({aum_str} from {files_str}).

{context_block}

{analytics_block}

Use the analyze_portfolio tool for ALL analytics questions.
  Call it with the appropriate question_type:
    • consolidated_view  → full portfolio table with values and gain/loss
    • biggest_gainers    → which holdings gained the most (% and $)
    • biggest_losers     → which holdings are at a loss
    • overweight         → which asset classes or positions are over/underweight
    • summary            → quick AUM and return snapshot
Use save_consolidated_portfolio to export an updated JSON for Notebook 13.
Use search_web for current market prices or news on any specific holding.
"""
    else:
        a2_ctx = """
No portfolio has been loaded. Ask the user to run Notebook 12 first,
or use search_web to answer general market questions.
"""

    a2 = f"""\
You are a Portfolio Consolidation & Analytics Specialist (Agent 2).
{a2_ctx}
Be concise, factual, and always remind users this is educational only.
"""

    # ── Agent 3: Education Agent ────────────────────────────────────────────
    if personalised:
        personalisation_note = (
            "Always relate explanations to the investor's actual holdings "
            "listed above. Reference real tickers and allocations when "
            "illustrating any concept."
        )
    else:
        personalisation_note = (
            "No portfolio is loaded. Provide general investment education. "
            "Encourage the user to run Notebook 12 for personalised explanations."
        )

    a3 = f"""\
You are an expert investment educator.

{context_block}

Teaching Framework — use this for EVERY concept explained:
1. Simple Definition  — one sentence, no jargon
2. Real-World Analogy — non-financial comparison
3. Portfolio Application — reference the investor's real tickers & allocations
4. Practical Takeaway  — one concrete action or consideration

{personalisation_note}
Use search_web only for current prices or recent news on a specific holding.
Always end with: 'This is for educational purposes only, not financial advice.'
"""

    return a1, a2, a3


AGENT1_SYSTEM, AGENT2_SYSTEM, AGENT3_SYSTEM = build_agent_prompts(portfolio_data)

print("All three system prompts built from loaded portfolio data")
print(f"  Personalised : {portfolio_data['personalised']}")
print(f"  Holdings     : {portfolio_data['metadata']['n_holdings']}")



All three system prompts built from loaded portfolio data
  Personalised : True
  Holdings     : 17


## Define All Tools

| Tool | Agent | Purpose |
|------|-------|---------|
| `search_web` | 1, 2, 3 | Current prices, news, fund data |
| `portfolio_generation` | 1 | Save a designed portfolio to JSON |
| `analyze_portfolio` | 2 | Return analytics from loaded portfolio |
| `save_consolidated_portfolio` | 2 | Export NB12-compatible JSON |


In [6]:
# ── Shared: Web Search ────────────────────────────────────────────────────────
@tool
def search_web(query: str) -> str:
    """Search the web for current market data, ETF info, or investment news."""
    try:
        return GoogleSerperAPIWrapper().run(query)
    except Exception as e:
        return f"Search unavailable: {e}"


# ── Agent 1: Portfolio Generation ─────────────────────────────────────────────
@tool
def portfolio_generation(portfolio_name: str, description: str,
                          holdings: list[dict]) -> str:
    """Save a designed portfolio to JSON.

    Call after the investor has approved the allocation.
    Holdings must include: ticker, company_name, allocation_pct, investment_type.
    Allocations must sum to ~100%.
    """
    if not holdings:
        return "No holdings provided."
    total = sum(h.get("allocation_pct", 0) for h in holdings)
    if not (98 <= total <= 102):
        return (f"Allocations sum to {total:.1f}% — must be ~100%. "
                "Adjust before saving.")
    portfolio = {"name": portfolio_name, "description": description,
                 "holdings": holdings}
    Path("../data/outputs").mkdir(parents=True, exist_ok=True)
    with open(AGENT1_OUTPUT_FILE, "w") as f:
        json.dump(portfolio, f, indent=2)
    SHARED_STATE["portfolio"] = portfolio
    return (f"Portfolio '{portfolio_name}' saved ({len(holdings)} holdings, "
            f"{total:.1f}% total). Agent 2 can now consolidate this.")


# ── Agent 2: Analyse loaded portfolio ─────────────────────────────────────────
# NOTE: query modes aligned with NB12 analyze_portfolio tool.
# Reads from portfolio_data["holdings"] which is loaded from NB12's
# consolidated_portfolio.json — includes gain_loss_usd and gain_loss_pct fields.
@tool
def analyze_portfolio(question_type: str = "consolidated_view") -> str:
    """Run analytics on the consolidated portfolio loaded from Notebook 12.

    question_type options (must match exactly):
        consolidated_view  — full holdings table with value, allocation, gain/loss
        biggest_gainers    — top 5 gainers by % and $, with portfolio total return
        biggest_losers     — all holdings at a loss, ranked worst first
        overweight         — asset-class balance vs targets + flagged positions
        summary            — AUM, return, top/bottom 3 in one snapshot
    """
    holdings = portfolio_data["holdings"]
    if not holdings:
        return ("No portfolio loaded. Run Notebook 12 and set "
                "CONSOLIDATED_PORTFOLIO_FILE correctly.")

    # Support gain/loss fields added in latest NB12 — safe .get() with fallback
    total_aum  = sum(h.get("value_usd", 0) or 0       for h in holdings)
    total_gain = sum(h.get("gain_loss_usd", 0) or 0   for h in holdings)
    total_cost = total_aum - total_gain
    total_ret  = round(total_gain / total_cost * 100, 2) if total_cost > 0 else 0
    q = question_type.lower().strip()

    # ── Consolidated view ──────────────────────────────────────────────────
    if q in ("consolidated_view", "consolidated", "view", "all"):
        rows = [{"ticker":          h["ticker"],
                 "type":            h.get("investment_type", "Unknown"),
                 "allocation_pct":  round(h.get("allocation_pct", 0), 2),
                 "value_usd":       round(h["value_usd"], 0) if h.get("value_usd") else None,
                 "gain_loss_usd":   round(h["gain_loss_usd"], 0) if h.get("gain_loss_usd") else None,
                 "gain_loss_pct":   h.get("gain_loss_pct"),
                 "sources":         h.get("source_files", [])}
                for h in holdings]
        type_totals = {}
        for h in holdings:
            t = h.get("investment_type", "Unknown")
            type_totals[t] = type_totals.get(t, 0) + h.get("allocation_pct", 0)
        return json.dumps({
            "total_holdings":       len(holdings),
            "total_aum":            round(total_aum, 0),
            "total_gain_usd":       round(total_gain, 0),
            "total_return_pct":     total_ret,
            "holdings":             rows,
            "asset_type_breakdown": {k: round(v, 2) for k, v in
                                     sorted(type_totals.items(), key=lambda x: -x[1])},
        }, indent=2)

    # ── Biggest gainers ────────────────────────────────────────────────────
    elif q in ("biggest_gainers", "gainers", "top_gainers", "gained_most"):
        gainers = sorted(
            [h for h in holdings if h.get("gain_loss_pct") is not None],
            key=lambda h: -(h["gain_loss_pct"] or 0)
        )
        top_dollar = max(holdings, key=lambda h: h.get("gain_loss_usd") or 0)
        return json.dumps({
            "top_5_by_pct": [{"ticker":    h["ticker"],
                               "gain_pct":  h["gain_loss_pct"],
                               "gain_usd":  round(h.get("gain_loss_usd") or 0, 0),
                               "value_usd": round(h.get("value_usd") or 0, 0),
                               "name":      h.get("company_name", "")}
                              for h in gainers[:5]],
            "top_gainer_by_pct":          gainers[0]["ticker"] if gainers else None,
            "top_gainer_by_usd":          top_dollar["ticker"],
            "top_gain_pct":               gainers[0]["gain_loss_pct"] if gainers else 0,
            "top_gain_usd":               round(top_dollar.get("gain_loss_usd") or 0, 0),
            "portfolio_total_gain_usd":   round(total_gain, 0),
            "portfolio_total_return_pct": total_ret,
        }, indent=2)

    # ── Biggest losers ─────────────────────────────────────────────────────
    elif q in ("biggest_losers", "losers", "lost_most", "losses"):
        losers = sorted(
            [h for h in holdings if (h.get("gain_loss_usd") or 0) < 0],
            key=lambda h: h.get("gain_loss_pct") or 0
        )
        total_loss = sum(h.get("gain_loss_usd") or 0 for h in losers)
        return json.dumps({
            "losers": [{"ticker":    h["ticker"],
                        "loss_pct":  h.get("gain_loss_pct"),
                        "loss_usd":  round(h.get("gain_loss_usd") or 0, 0),
                        "value_usd": round(h.get("value_usd") or 0, 0),
                        "name":      h.get("company_name", "")}
                       for h in losers],
            "total_holdings_at_loss":    len(losers),
            "total_holdings":            len(holdings),
            "total_loss_usd":            round(total_loss, 0),
            "biggest_loser_by_pct":      losers[0]["ticker"] if losers else None,
            "portfolio_net_gain_usd":    round(total_gain, 0),
            "portfolio_net_return_pct":  total_ret,
        }, indent=2)

    # ── Overweight / underweight ───────────────────────────────────────────
    elif q in ("overweight", "underweight", "rebalancing", "balance", "weight"):
        TARGETS = {"ETF": 45.0, "Stock": 25.0, "Bond ETF": 20.0,
                   "Mutual Fund": 5.0, "Other": 5.0}
        actual = {}
        for h in holdings:
            t = h.get("investment_type", "Unknown")
            actual[t] = actual.get(t, 0) + h.get("allocation_pct", 0)
        type_analysis = {}
        for t, tgt in TARGETS.items():
            act  = round(actual.get(t, 0), 2)
            diff = round(act - tgt, 2)
            type_analysis[t] = {
                "actual_pct": act, "target_pct": tgt, "diff_pct": diff,
                "status": ("OVERWEIGHT"     if diff >  5 else
                           "UNDERWEIGHT"    if diff < -5 else
                           "slightly_over"  if diff >  2 else
                           "slightly_under" if diff < -2 else "on_target"),
            }
        flagged = []
        for h in holdings:
            a = h.get("allocation_pct", 0)
            if   a > 15:  flag = "SIGNIFICANTLY_OVERWEIGHT"
            elif a > 10:  flag = "OVERWEIGHT"
            elif a < 1.0: flag = "VERY_SMALL_POSITION"
            else: continue
            flagged.append({"ticker": h["ticker"], "allocation_pct": round(a, 2),
                             "value_usd": round(h.get("value_usd") or 0, 0),
                             "flag": flag})
        return json.dumps({
            "asset_class_analysis": type_analysis,
            "flagged_positions":    flagged,
            "tech_concentration_pct": round(sum(
                h.get("allocation_pct", 0) for h in holdings
                if h["ticker"] in ("NVDA", "MSFT", "AAPL", "GOOG")), 2),
        }, indent=2)

    # ── Summary (default) ──────────────────────────────────────────────────
    else:
        top3 = sorted([h for h in holdings if h.get("gain_loss_pct") is not None],
                      key=lambda h: -(h["gain_loss_pct"] or 0))[:3]
        bot3 = sorted([h for h in holdings if (h.get("gain_loss_usd") or 0) < 0],
                      key=lambda h: h.get("gain_loss_pct") or 0)[:3]
        return json.dumps({
            "total_aum":        round(total_aum, 0),
            "total_gain_usd":   round(total_gain, 0),
            "total_return_pct": total_ret,
            "total_holdings":   len(holdings),
            "top_3_gainers":    [{"ticker": h["ticker"],
                                  "gain_pct": h["gain_loss_pct"]} for h in top3],
            "top_3_losers":     [{"ticker": h["ticker"],
                                  "loss_pct": h["gain_loss_pct"]} for h in bot3],
        }, indent=2)


# ── Agent 2: Save consolidated portfolio ──────────────────────────────────────
@tool
def save_consolidated_portfolio(notes: str = "") -> str:
    """Save the current consolidated portfolio to disk in NB12-compatible format.
    Overwrites consolidated_portfolio.json so Notebook 13 picks it up.
    """
    holdings = portfolio_data["holdings"]
    metadata = portfolio_data["metadata"]
    if not holdings:
        return "No portfolio loaded — nothing to save."
    output = {
        "source_folder":   PORTFOLIO_FOLDER,
        "source_files":    metadata["source_files"],
        "total_holdings":  metadata["n_holdings"],
        "total_allocation": round(sum(h.get("allocation_pct", 0) for h in holdings), 4),
        "total_value_usd": metadata.get("total_value"),
        "holdings":        holdings,
    }
    if notes:
        output["notes"] = notes
    out_path = Path(CONSOLIDATED_PORTFOLIO_FILE)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with open(out_path, "w") as f:
        json.dump(output, f, indent=2, default=str)
    return (f"Saved {len(holdings)} holdings to {out_path}. "
            "Notebook 13 will load this on next run.")


# ── Tool sets per agent ────────────────────────────────────────────────────────
AGENT1_TOOLS = [search_web, portfolio_generation]
AGENT2_TOOLS = [search_web, analyze_portfolio, save_consolidated_portfolio]
AGENT3_TOOLS = [search_web]

print(f"Agent 1 tools: {[t.name for t in AGENT1_TOOLS]}")
print(f"Agent 2 tools: {[t.name for t in AGENT2_TOOLS]}")
print(f"Agent 3 tools: {[t.name for t in AGENT3_TOOLS]}")



Agent 1 tools: ['search_web', 'portfolio_generation']
Agent 2 tools: ['search_web', 'analyze_portfolio', 'save_consolidated_portfolio']
Agent 3 tools: ['search_web']


## Router — Decides Which Agent(s) to Invoke

A lightweight LLM call (no tools) that classifies the user's query and
returns a route string such as `agent3` or `agent2+agent3`.


In [7]:
ROUTER_PROMPT = """\
You are a router for a 3-agent investment system.
Classify the user's message and respond with ONLY one of these exact strings:

  agent1               → User wants to build or design a new portfolio from scratch
  agent2               → User asks portfolio analytics: consolidated view, gainers,
                         losers, overweight/underweight, allocation, concentration
  agent3               → User asks to explain or learn an investment concept
                         (e.g. 'what is beta', 'explain ETF', 'what is DCA')
  agent1+agent2        → User gives a profile AND wants analytics right away
  agent2+agent3        → User asks analytics AND wants a plain-English explanation
  agent1+agent2+agent3 → Full pipeline: build, consolidate, then educate

Examples:
  'Help me build a long-term portfolio.'                    → agent1
  'I am 45, moderate risk, retiring in 15 years'            → agent1
  'What is beta?'                                           → agent3
  'What is dollar-cost averaging?'                          → agent3
  'Please provide a consolidated view of my portfolio'      → agent2
  'Which stock or ETF have I gained the most'               → agent2
  'Which stock or ETF have I lost the most'                 → agent2
  'Which stocks are overweight or underweight'              → agent2+agent3
  'How volatile is my portfolio?'                           → agent2+agent3
  'Show me my asset allocation'                             → agent2
  'Should I rebalance?'                                     → agent2+agent3
"""

router_llm = load_llm_from_env()

VALID_ROUTES = {
    "agent1", "agent2", "agent3",
    "agent1+agent2", "agent2+agent3", "agent1+agent2+agent3",
}


def route_query(query: str) -> str:
    """Classify a user query and return the route string."""
    resp  = router_llm.invoke([
        SystemMessage(content=ROUTER_PROMPT),
        HumanMessage(content=query),
    ])
    route = resp.content.strip().lower()
    return route if route in VALID_ROUTES else "agent3"


# ── Quick self-test ───────────────────────────────────────────────────────────
print("Router test:")
for q in [
    "Help me build a long-term portfolio.",
    "What is beta?",
    "How volatile is my portfolio?",
    "Show me my asset allocation",
    "What is dollar-cost averaging?",
]:
    print(f"  {route_query(q):<28}  ← '{q}'")



🤖 Loading LLM: openai / gpt-4.1-mini
Router test:
  agent1                        ← 'Help me build a long-term portfolio.'
  agent3                        ← 'What is beta?'
  agent2+agent3                 ← 'How volatile is my portfolio?'
  agent2                        ← 'Show me my asset allocation'
  agent3                        ← 'What is dollar-cost averaging?'


## Build Individual Agent Graphs

In [8]:
def build_agent_graph(system_prompt: str, agent_tools: list):
    """
    Factory: build a standard tool-calling LangGraph agent.

    The system_prompt is captured in the closure so the graph always uses
    the prompt that was active when it was compiled.  Call build_agent_graph
    again (via reload_agents()) if you reload the portfolio.
    """
    llm_with_tools = load_llm_from_env().bind_tools(agent_tools)

    def assistant_node(state: MessagesState):
        msgs = [SystemMessage(content=system_prompt)] + state["messages"]
        return {"messages": [llm_with_tools.invoke(msgs)]}

    def route_node(state: MessagesState) -> Literal["tools", "__end__"]:
        return "tools" if state["messages"][-1].tool_calls else "__end__"

    g = StateGraph(MessagesState)
    g.add_node("assistant", assistant_node)
    g.add_node("tools", ToolNode(agent_tools))
    g.add_edge(START, "assistant")
    g.add_conditional_edges("assistant", route_node, ["tools", "__end__"])
    g.add_edge("tools", "assistant")
    return g.compile(checkpointer=MemorySaver())


def build_all_agents():
    """Compile all three agent graphs from current system prompts."""
    a1 = build_agent_graph(AGENT1_SYSTEM, AGENT1_TOOLS)
    a2 = build_agent_graph(AGENT2_SYSTEM, AGENT2_TOOLS)
    a3 = build_agent_graph(AGENT3_SYSTEM, AGENT3_TOOLS)
    print("Agent 1 (Portfolio Designer)      compiled")
    print("Agent 2 (Consolidator & Analyst)  compiled")
    print("Agent 3 (Investment Educator)     compiled")
    return a1, a2, a3


agent1_graph, agent2_graph, agent3_graph = build_all_agents()


🤖 Loading LLM: openai / gpt-4.1-mini
🤖 Loading LLM: openai / gpt-4.1-mini
🤖 Loading LLM: openai / gpt-4.1-mini
Agent 1 (Portfolio Designer)      compiled
Agent 2 (Consolidator & Analyst)  compiled
Agent 3 (Investment Educator)     compiled


## Sequential Orchestrator

Routes each query then runs agents in order, passing each agent's response
as context to the next agent in the chain.


In [9]:
def run_agent(graph, user_input: str, thread_id: str,
              context_prefix: str = "") -> str:
    """Run one agent and return its final text response."""
    full_input = (f"{context_prefix}\n\n{user_input}".strip()
                  if context_prefix else user_input)
    config = {"configurable": {"thread_id": thread_id}}
    result = None
    for event in graph.stream(
        {"messages": [HumanMessage(content=full_input)]},
        config,
        stream_mode="values",
    ):
        result = event
    return result["messages"][-1].content if result else ""


def sequential_chat(user_input: str,
                    session_id: str = "seq",
                    verbose: bool = True) -> str:
    """
    Main entry point for the Sequential Multi-Agent System.

    Workflow:
        1. Router classifies the query → route string
        2. Agents run in sequence (1 → 2 → 3 as needed)
        3. Each agent receives the prior agent's full response as context
        4. Final agent's response is returned

    Args:
        user_input : The user's question
        session_id : Thread ID for conversation memory (pass the same ID
                     for multi-turn conversations with the same agent)
        verbose    : Print routing decisions and per-agent previews

    Returns:
        Final agent response as a string
    """
    route = route_query(user_input)
    if verbose:
        print(f"  Route: {route}")

    graph_map = {
        "agent1": agent1_graph,
        "agent2": agent2_graph,
        "agent3": agent3_graph,
    }
    label_map = {
        "agent1": "Agent 1 (Designer)",
        "agent2": "Agent 2 (Analyst)",
        "agent3": "Agent 3 (Educator)",
    }

    agents_to_run = route.split("+")
    last_response = ""
    context       = ""

    for i, agent_name in enumerate(agents_to_run):
        agent_name = agent_name.strip()
        graph  = graph_map.get(agent_name)
        label  = label_map.get(agent_name, agent_name)
        if not graph:
            continue

        if verbose:
            print(f"  ▶ Running {label}...")

        thread_id  = f"{session_id}_{agent_name}"
        ctx_prefix = (f"[Context from {label_map.get(agents_to_run[i-1].strip(), 'previous agent')}]:\n"
                      f"{context[:1200]}\n[End context]") if context and i > 0 else ""

        response      = run_agent(graph, user_input, thread_id, ctx_prefix)
        last_response = response
        context       = response

        if verbose:
            preview = response[:200].replace("\n", " ")
            suffix  = "..." if len(response) > 200 else ""
            print(f"  ✓ {label}: {preview}{suffix}")

    SHARED_STATE["conversation_history"].append({
        "input": user_input, "route": route, "response": last_response
    })
    return last_response


print("Sequential Orchestrator ready")
print("Usage: sequential_chat('your question', session_id='my_session')")


Sequential Orchestrator ready
Usage: sequential_chat('your question', session_id='my_session')


## Reload Portfolio (optional)

If you re-run Notebook 12 and want to pick up an updated
`consolidated_portfolio.json` **without restarting the kernel**, run this cell.
It reloads the file, rebuilds all three system prompts, and recompiles the agents.


In [10]:
def reload_agents():
    """Hot-reload portfolio from NB12 and recompile all three agents."""
    global portfolio_data, AGENT1_SYSTEM, AGENT2_SYSTEM, AGENT3_SYSTEM
    global agent1_graph, agent2_graph, agent3_graph

    portfolio_data = load_portfolio_context(CONSOLIDATED_PORTFOLIO_FILE)
    AGENT1_SYSTEM, AGENT2_SYSTEM, AGENT3_SYSTEM = build_agent_prompts(portfolio_data)
    agent1_graph, agent2_graph, agent3_graph     = build_all_agents()

    print(DIVIDER)
    print("Portfolio reloaded — all agents recompiled")
    print(DIVIDER)
    print(portfolio_data["context_block"])
    print(DIVIDER)

# Uncomment to reload:
# reload_agents()


---
## Test Queries — Sequential Multi-Agent System

Five queries that exercise the full routing logic.
Each runs in its own `session_id` so conversation history does not bleed between tests.

| # | Query | Expected Route |
|---|-------|---------------|
| 1 | Help me build a long-term portfolio | `agent1` |
| 2 | I am 45, moderate risk, retiring in 15 years | `agent1` |
| 3 | What is beta? | `agent3` |
| 4 | What is dollar-cost averaging? | `agent3` |
| 5 | How volatile is my portfolio? | `agent2+agent3` |


In [11]:
q1 = "Help me build a long-term portfolio."
print(DIVIDER)
print(f"TEST QUERY 1: {q1}")
print(DIVIDER)
print(f"User: {q1}\n")
response1 = sequential_chat(q1, session_id="test1")
print(f"\nFinal Response:\n{response1}")

TEST QUERY 1: Help me build a long-term portfolio.
User: Help me build a long-term portfolio.

  Route: agent1
  ▶ Running Agent 1 (Designer)...
  ✓ Agent 1 (Designer): I'd be happy to help you build a long-term portfolio! To get started, could you tell me a bit about your primary investment goal? For example, are you investing for retirement, to purchase something i...

Final Response:
I'd be happy to help you build a long-term portfolio! To get started, could you tell me a bit about your primary investment goal? For example, are you investing for retirement, to purchase something in the future, growth, income, or education? Also, what's your level of investing experience—beginner, intermediate, or advanced?


In [12]:
q2 = "I am 45, moderate risk, retiring in 15 years. Help me plan."
print(DIVIDER)
print(f"TEST QUERY 2: {q2}")
print(DIVIDER)
print(f"User: {q2}\n")
response2 = sequential_chat(q2, session_id="test2")
print(f"\nFinal Response:\n{response2}")

TEST QUERY 2: I am 45, moderate risk, retiring in 15 years. Help me plan.
User: I am 45, moderate risk, retiring in 15 years. Help me plan.

  Route: agent1
  ▶ Running Agent 1 (Designer)...
  ✓ Agent 1 (Designer): It's great to hear that you're thinking about planning for your retirement! To tailor the best investment strategy for you, I'd like to understand a bit more about your experience and preferences.  Co...

Final Response:
It's great to hear that you're thinking about planning for your retirement! To tailor the best investment strategy for you, I'd like to understand a bit more about your experience and preferences.

Could you please tell me:
1. How would you characterize your investing experience? Are you a beginner, intermediate, or advanced investor?
2. How do you usually react to a significant market drop, say about 15%? Do you tend to sell all, sell some, hold, or even buy more?


In [13]:
q3 = "What is beta?"
print(DIVIDER)
print(f"TEST QUERY 3: {q3}")
print(DIVIDER)
print(f"User: {q3}\n")
response3 = sequential_chat(q3, session_id="test3")
print(f"\nFinal Response:\n{response3}")

TEST QUERY 3: What is beta?
User: What is beta?

  Route: agent3
  ▶ Running Agent 3 (Educator)...
  ✓ Agent 3 (Educator): 1. Simple Definition — Beta is a number that shows how much a stock or portfolio's price moves compared to the overall market.  2. Real-World Analogy — Imagine beta like a car's sensitivity to the gas...

Final Response:
1. Simple Definition — Beta is a number that shows how much a stock or portfolio's price moves compared to the overall market.

2. Real-World Analogy — Imagine beta like a car's sensitivity to the gas pedal: a high beta car speeds up and slows down a lot with small pedal changes, while a low beta car changes speed more gently.

3. Portfolio Application — For example, NVIDIA (NVDA) and Tesla (TSLA), being individual stocks, tend to have higher betas and may move more dramatically than broad-market ETFs like Vanguard Total Stock Market ETF (VTI), which represents the whole market and has a beta close to 1.

4. Practical Takeaway — If you want a more 

In [14]:
q4 = "What is dollar-cost averaging?"
print(DIVIDER)
print(f"TEST QUERY 4: {q4}")
print(DIVIDER)
print(f"User: {q4}\n")
response4 = sequential_chat(q4, session_id="test4")
print(f"\nFinal Response:\n{response4}")

TEST QUERY 4: What is dollar-cost averaging?
User: What is dollar-cost averaging?

  Route: agent3
  ▶ Running Agent 3 (Educator)...
  ✓ Agent 3 (Educator): Certainly! Here's an explanation of dollar-cost averaging using the teaching framework:  1. Simple Definition: Dollar-cost averaging is an investment strategy where you invest a fixed amount of money ...

Final Response:
Certainly! Here's an explanation of dollar-cost averaging using the teaching framework:

1. Simple Definition:
Dollar-cost averaging is an investment strategy where you invest a fixed amount of money at regular intervals, regardless of the asset's price.

2. Real-World Analogy:
Imagine buying coffee every Monday with the same $5. Some weeks the coffee might cost $4, other weeks $6, so you get varying amounts of coffee for the same money. Over time, you balance out the price you pay.

3. Portfolio Application:
If you regularly invest $1,000 each month into your Vanguard Total Stock Market ETF (VTI) holding, you buy 

In [15]:
q5 = "How volatile is my portfolio and what does that mean?"
print(DIVIDER)
print(f"TEST QUERY 5: {q5}")
print(DIVIDER)
print(f"User: {q5}\n")
response5 = sequential_chat(q5, session_id="test5")
print(f"\nFinal Response:\n{response5}")

TEST QUERY 5: How volatile is my portfolio and what does that mean?
User: How volatile is my portfolio and what does that mean?

  Route: agent2+agent3
  ▶ Running Agent 2 (Analyst)...
  ✓ Agent 2 (Analyst): Your portfolio has generated a total return of 16.01%, with a gain of about $135,237 on a total AUM of $979,963. The top gainers in your portfolio show high volatility assets such as NVIDIA (NVDA) wit...
  ▶ Running Agent 3 (Educator)...
  ✓ Agent 3 (Educator): 1. Simple Definition — Volatility is how much and how quickly the value of your investments can go up and down.  2. Real-World Analogy — Think of volatility like the waves in the ocean: sometimes they...

Final Response:
1. Simple Definition — Volatility is how much and how quickly the value of your investments can go up and down.

2. Real-World Analogy — Think of volatility like the waves in the ocean: sometimes they are calm and steady (low volatility), and sometimes they are rough and choppy with big waves (high volatilit

## Session Summary

In [ ]:
print(DIVIDER)
print("SESSION SUMMARY — All 5 Test Queries")
print(DIVIDER)
print(f"{'#':<3} {'Route':<28} {'Query'}")
print("-" * 70)
for i, entry in enumerate(SHARED_STATE["conversation_history"], 1):
    query_preview = entry["input"][:45]
    print(f"{i:<3} {entry['route']:<28} {query_preview}")
print(DIVIDER)


## Interactive Mode — Full Sequential System

All three agents are live. The router decides which to call based on your question.
Type `quit` to stop.


In [ ]:
print(DIVIDER)
print("SEQUENTIAL MULTI-AGENT SYSTEM — Interactive Mode")
print(DIVIDER)
if portfolio_data["personalised"]:
    tickers = [h["ticker"] for h in portfolio_data["holdings"]]
    print(f"Portfolio: {', '.join(tickers[:8])}"
          + (f" ... (+{len(tickers)-8} more)" if len(tickers) > 8 else ""))
else:
    print("GENERIC MODE — run Notebook 12 for personalised responses")
print()
print("Try: 'What is alpha?' / 'Show my asset allocation' / 'Build me a portfolio'")
print("     'What is rebalancing?' / 'How volatile is my portfolio?'")
print("Type 'quit' to stop.")
print(DIVIDER)

while True:
    user_input = input("\nYou: ").strip()
    if not user_input:
        continue
    if user_input.lower() in ("quit", "exit", "q"):
        print("\nSession ended.")
        break
    response = sequential_chat(user_input, session_id="interactive")
    print(f"\nAgent: {response}")


---
## Summary: What Changed and Architecture Notes

### What Changed from Previous Version

| Area | Before | After |
|------|--------|-------|
| Portfolio data | 25-row `SAMPLE_PORTFOLIOS` hardcoded in cell | Loaded from `consolidated_portfolio.json` (NB12 output) |
| Analytics engine | Duplicated `consolidate_holdings` + `compute_analytics` | Reads pre-computed data from NB12 JSON |
| Agent 2 prompt | Hardcoded `"~$1,026,834 AUM across 5 accounts"` | Built at runtime from `load_portfolio_context()` |
| Agent 3 prompt | Hardcoded ticker list | Built at runtime — same pattern as NB13 |
| `analyze_portfolio` tool | Ran analytics on `SAMPLE_PORTFOLIOS` constant | Queries `portfolio_data["holdings"]` loaded from file |
| `save_consolidated_portfolio` | Wrote custom Excel+JSON format | Writes NB12-compatible schema (Notebook 13 reads it) |
| Install cell | Included `pypdf`, `pillow` (not needed) | Replaced with `PyMuPDF` (consistent with NB12) |

### Contract Between Notebooks

```
Notebook 11 (Agent 1 — Designer)
    └── ../data/outputs/portfolio.json
                │
Notebook 12 (Agent 2 — Consolidator)
    reads: PORTFOLIO_FOLDER (any number of files)
    └── ../data/outputs/consolidated_portfolio.json
                │
Notebook 13 (Agent 3 — Educator)          Notebook 14 (Orchestrator)
    reads: consolidated_portfolio.json   ← reads: consolidated_portfolio.json
    └── Education chat                       └── Routes + chains 1→2→3
```

### Architecture
```
User Query
    │
    ▼
┌──────────────────┐
│  Router (LLM)    │  → route string (no tools, lightweight)
└────────┬─────────┘
         │
┌────────▼───────────────────────────────────────┐
│           Sequential Orchestrator               │
│                                                 │
│  agents_to_run = route.split("+")               │
│  for each agent:                                │
│      response = run_agent(graph, input, ctx)    │
│      ctx = response  (pass to next agent)       │
└────────────────────────────────────────────────┘
         │
  ┌──────┼──────┐
  ▼      ▼      ▼
Agent1 Agent2 Agent3
  │      │      │
 NB11   NB12   NB13
schema schema schema
```

### Remaining Limitations
- Each agent has its own `MemorySaver` — no shared cross-agent message history.
  For a fully unified memory store, migrate to a LangGraph Supervisor pattern
  (Notebook 15).
- Agent 1 is a cold-start profiler. It asks questions before generating a
  portfolio — use the interactive mode with a persistent `session_id` for
  multi-turn profiling sessions.
- Context between agents is capped at 1200 characters to avoid token limits.
  For very detailed analytics passes, pass only the relevant sub-section.
